# Mean Reversion — equilibrium window sweep

Tests Carver's fast mean-reversion rule across different equilibrium EWMA windows.
Longer `equil_span` = slower mean reversion (fades larger, longer swings).

The forecast scalar must be **recalibrated for each window** (the raw forecast magnitude
depends on the window), so each variation is calibrated on this basket with
`ignore_zeros=True` (the trend overlay zeroes ~60% of the signal).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "sysstrat").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / "data"
print(f"Project root: {project_root}")

from sysstrat.data import load_simple_price_csv
from sysstrat.core import Asset, Capital, FixedRiskSizer
from sysstrat.engine import BacktestRunner, PortfolioRunner
from sysstrat.strategies import MeanReversionStrategy
from sysstrat.strategies.transforms import mean_reversion_forecast, calibrate_forecast_scalar

## 1. Load instruments (common period)

In [ ]:
INSTRUMENTS = {
    "MCFTR": "MCFTR.csv", "RGBITR": "RGBITR.csv", "GLDRUB": "GLDRUB_TOM.csv",
    "CNYRUB": "CNYRUB_TOM.csv", "USDRUB": "USDRUB.csv",
}
assets = {
    t: Asset(ticker=t, price_data=load_simple_price_csv(DATA_DIR / f), commission_rate=0.0004, slippage_rate=0.001)
    for t, f in INSTRUMENTS.items()
}
start = max(a.price_data.index.min() for a in assets.values())
end = min(a.price_data.index.max() for a in assets.values())
assets = {t: a.slice(start, end) for t, a in assets.items()}
print(f"Common period: {start.date()} -> {end.date()}")

## 2. Sweep equilibrium windows

For each `equil_span`, calibrate the forecast scalar on the basket, then build an
equal-weight portfolio across the 5 instruments.

In [ ]:
import pandas as pd

CAPITAL = 100_000
capital = Capital(initial_capital=CAPITAL)
sizer = FixedRiskSizer(risk_target=0.20, max_leverage=1.0)

def run(equil_span):
    # 1. Calibrate the scalar on this basket for this window.
    raw = [mean_reversion_forecast(a.price_data, equil_span=equil_span).dropna() for a in assets.values()]
    scalar = calibrate_forecast_scalar(raw, target=1.0, clip=2.0, ignore_zeros=True)
    # 2. Run the strategy (equal-weight portfolio).
    strategy = MeanReversionStrategy(equil_span=equil_span, forecast_scalar=scalar)
    reports = {t: BacktestRunner(capital, a, sizer).run(strategy) for t, a in assets.items()}
    port = PortfolioRunner(capital).run(reports)
    return scalar, port.metrics

rows = []
for es in [5, 10, 20, 40, 80, 120]:
    scalar, m = run(es)
    rows.append({
        "equil_span": es,
        "scalar": round(scalar, 4),
        "sharpe": round(m.sharpe_ratio, 2),
        "vol %": round(m.annual_volatility_pct, 2),
        "max DD %": round(m.max_drawdown_pct, 1),
        "total ret %": round(m.total_return_pct, 1),
    })

print(pd.DataFrame(rows).set_index("equil_span").to_string())

> Note: on this 5-instrument RUB basket the mean-reversion rule is a weak-to-negative
> standalone performer (strong trends + violent reversals leave it with little to fade).
> Its value is diversification in a combined forecast, not standalone Sharpe.